## 08 - BiLSTM with Token Embeddings for BFRB Detection (Milestone 2)

In this notebook I implement a Bidirectional LSTM model that works on tokenized sensor sequences.
Instead of feeding raw float values into the model, we first convert sensor readings into
discrete tokens in notebook 06, then learn an embedding for each token, and finally pass the
sequence of embeddings through a BiLSTM classifier.

We chose BiLSTM because it reads the sequence in both directions, which helps capture context
from both before and after each timestep. This is a natural fit for gesture data where the
middle of a movement depends on what came before and after it.

What makes this different from the paper's CNN-BiLSTM is that the paper's model works on raw
TOF float matrices, while this notebook works on learned token embeddings, which is the NLP framing.

### Inputs (from `06_NLPDataPrep.ipynb`)
- `../data/processed/train_text_seq.pkl`
- `../data/processed/test_text_seq.pkl`
- `../data/processed/nlp_vocab.pkl`
- `../data/processed/nlp_max_seq_len.pkl`

### Output
- `../results/bilstm_results.pkl` - binary F1, macro F1, predictions, labels, and hyperparameters

## 1. Setup and Imports

This cell imports the libraries used throughout the notebook, sets the random seed for
reproducibility, selects CPU or GPU automatically, and defines the data and results paths.

In [ ]:

import pickle
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.metrics import f1_score
from pathlib import Path

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PROCESSED = Path("../data/processed")
RESULTS_DIR    = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print("Imports done.")

Device: cpu
Imports done.


## 2. Load Preprocessed Data

Loading the token ID sequences that were prepared in the NLP preprocessing notebook.
Each sequence is a list of integer IDs corresponding to sensor tokens.

In [ ]:
with open(DATA_PROCESSED / "train_text_seq.pkl", "rb") as f:
    train_data = pickle.load(f)

with open(DATA_PROCESSED / "test_text_seq.pkl", "rb") as f:
    test_data = pickle.load(f)

with open(DATA_PROCESSED / "nlp_vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

with open(DATA_PROCESSED / "nlp_max_seq_len.pkl", "rb") as f:
    max_seq_len = pickle.load(f)

VOCAB_SIZE = len(vocab)
PAD_IDX    = vocab["<PAD>"]

print(f"Train sequences : {len(train_data)}")
print(f"Test sequences  : {len(test_data)}")
print(f"Vocabulary size : {VOCAB_SIZE}")
print(f"Max seq length  : {max_seq_len}")
print(f"PAD index       : {PAD_IDX}")

Train sequences : 6515
Test sequences  : 1629
Vocabulary size : 38
Max seq length  : 8400
PAD index       : 0


## 3. Hyperparameters

I truncate sequences to 512 tokens since the full sequences (~8400 tokens) would be very
slow to train on CPU. The embedding dimension is 64 and I use 2 stacked BiLSTM layers
with a hidden size of 128 per direction (256 total after concatenation).

In [ ]:
MAX_LEN     = 512  

# Model architecture
EMBED_DIM   = 64   
HIDDEN_DIM  = 128  
NUM_LAYERS  = 2    
DROPOUT     = 0.3

# Training
BATCH_SIZE  = 32
EPOCHS      = 10
LR          = 0.001

print("Hyperparameters set.")
print(f"  Truncation length : {MAX_LEN}")
print(f"  Embedding dim     : {EMBED_DIM}")
print(f"  BiLSTM hidden dim : {HIDDEN_DIM} (x2 bidirectional = {HIDDEN_DIM * 2})")
print(f"  BiLSTM layers     : {NUM_LAYERS}")
print(f"  Batch size        : {BATCH_SIZE}")
print(f"  Epochs            : {EPOCHS}")

Hyperparameters set.
  Truncation length : 512
  Embedding dim     : 64
  BiLSTM hidden dim : 128 (x2 bidirectional = 256)
  BiLSTM layers     : 2
  Batch size        : 32
  Epochs            : 10


## 4. Dataset and DataLoader

The dataset class truncates each token sequence and also returns the actual
sequence length before padding. We need the lengths so we can use
`pack_padded_sequence` later which tells the LSTM to ignore padding tokens.

In [ ]:
class BFRBTokenDataset(Dataset):
    def __init__(self, records, max_len):
        self.records = records
        self.max_len = max_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record    = self.records[idx]
        token_ids = record["token_ids"][:self.max_len]
        label     = record["label"]
        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(len(token_ids), dtype=torch.long),  # actual length before padding
            torch.tensor(label, dtype=torch.long)
        )


def collate_fn(batch):
    sequences, lengths, labels = zip(*batch)
    padded  = pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    lengths = torch.stack(lengths)
    labels  = torch.stack(labels)
    return padded, lengths, labels


train_dataset = BFRBTokenDataset(train_data, MAX_LEN)
test_dataset  = BFRBTokenDataset(test_data,  MAX_LEN)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches : {len(train_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 204
Test batches  : 51


### Model Architecture

The model has four parts:
1.⁠ ⁠An embedding layer that maps each token ID to a 64-dim vector
2.⁠ ⁠A 2-layer bidirectional LSTM
3.⁠ ⁠Mean pooling over all non-padding timesteps
4.⁠ ⁠A linear classifier head

I use mean pooling instead of just taking the last hidden state because gesture sequences can be long and the relevant signal is spread across the whole sequence rather than concentrated at the end.

In [ ]:
class BiLSTMBFRBClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers,
                 num_classes=2, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.bilstm    = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        emb = self.embedding(x)           # (batch, seq_len, embed_dim)

        packed = pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        output_packed, _ = self.bilstm(packed)
        output, _        = pad_packed_sequence(output_packed, batch_first=True)

        mask    = (x != PAD_IDX).unsqueeze(-1).float()   # (batch, seq_len, 1)
        summed  = (output * mask).sum(dim=1)              # (batch, hidden_dim * 2)
        lengths_expanded = lengths.unsqueeze(-1).float().to(summed.device)
        pooled  = summed / lengths_expanded.clamp(min=1)  # (batch, hidden_dim * 2)

        pooled = self.dropout(pooled)
        return self.classifier(pooled)   


model = BiLSTMBFRBClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    num_classes=2,
    dropout=DROPOUT,
    pad_idx=PAD_IDX
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,}")
print(model)

Model parameters: 596,866
BiLSTMBFRBClassifier(
  (embedding): Embedding(38, 64, padding_idx=0)
  (bilstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=256, out_features=2, bias=True)
)


## 6. Training

Training uses cross-entropy loss with the Adam optimizer. Each epoch iterates over mini-batches
of padded token sequences, computes predictions, backpropagates the loss, and reports the
average training loss and accuracy.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for token_ids, lengths, labels in loader:
        token_ids = token_ids.to(DEVICE)
        labels    = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(token_ids, lengths)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / total, correct / total


print("Starting training...")
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    print(f"Epoch {epoch:02d}/{EPOCHS} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

print("Training complete.")

Starting training...
Epoch 01/10 | Loss: 0.4583 | Train Acc: 0.7820
Epoch 02/10 | Loss: 0.3844 | Train Acc: 0.8332
Epoch 03/10 | Loss: 0.3479 | Train Acc: 0.8436
Epoch 04/10 | Loss: 0.3254 | Train Acc: 0.8533
Epoch 05/10 | Loss: 0.3016 | Train Acc: 0.8660
Epoch 06/10 | Loss: 0.2958 | Train Acc: 0.8666
Epoch 07/10 | Loss: 0.2780 | Train Acc: 0.8734
Epoch 08/10 | Loss: 0.2676 | Train Acc: 0.8794
Epoch 09/10 | Loss: 0.2540 | Train Acc: 0.8838
Epoch 10/10 | Loss: 0.2498 | Train Acc: 0.8856
Training complete.


## 7. Evaluation

Computing the two required metrics: binary F1 for BFRB vs non-BFRB and macro-averaged F1
across the evaluation labels.

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for token_ids, lengths, labels in loader:
            token_ids = token_ids.to(DEVICE)
            logits    = model(token_ids, lengths)
            preds     = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)


preds, labels = evaluate(model, test_loader)

binary_f1 = f1_score(labels, preds, average="binary", pos_label=1)
macro_f1  = f1_score(labels, preds, average="macro")

print(f"Binary F1-score        : {binary_f1 * 100:.2f}%")
print(f"Macro-averaged F1-score: {macro_f1  * 100:.2f}%")

Binary F1-score        : 89.98%
Macro-averaged F1-score: 84.19%


## 8. Save Results

This final step stores the evaluation metrics, predictions, labels, and key hyperparameters
in a pickle file so the results can be reused later for comparison and reporting. 

In [ ]:
results = {
    "model"     : "BiLSTM",
    "binary_f1" : binary_f1,
    "macro_f1"  : macro_f1,
    "preds"     : preds,
    "labels"    : labels,
    "hyperparams": {
        "embed_dim" : EMBED_DIM,
        "hidden_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "max_len"   : MAX_LEN,
        "batch_size": BATCH_SIZE,
        "epochs"    : EPOCHS,
        "lr"        : LR
    }
}

with open(RESULTS_DIR / "bilstm_results.pkl", "wb") as f:
    pickle.dump(results, f)

print("Saved: ../results/bilstm_results.pkl")
print("\n=== BiLSTM Model Complete ===")
print(f"Binary F1  : {binary_f1 * 100:.2f}%")
print(f"Macro F1   : {macro_f1  * 100:.2f}%")

Saved: ../results/bilstm_results.pkl

=== BiLSTM Model Complete ===
Binary F1  : 89.98%
Macro F1   : 84.19%
